## ETL ODEPA

In [7]:
# ============================================================
# 1. SUBIR ARCHIVO ODEPA
# ============================================================
from google.colab import files
import pandas as pd
import numpy as np

print("Archivo de ODEPA (Catastro Frutícola):")
uploaded = files.upload()

# Detectar nombre del archivo subido
nombre_archivo = list(uploaded.keys())[0]
print(f"\nArchivo subido: {nombre_archivo}")

Archivo de ODEPA (Catastro Frutícola):


Saving a1a3ffab-f3fd-4928-8948-c1bd9ba82d9c.csv to a1a3ffab-f3fd-4928-8948-c1bd9ba82d9c.csv

Archivo subido: a1a3ffab-f3fd-4928-8948-c1bd9ba82d9c.csv


In [8]:
# ============================================================
# 2. CARGAR EL CSV
# ============================================================
for enc in ['utf-8-sig', 'utf-8', 'latin-1']:
    try:
        df = pd.read_csv(nombre_archivo, encoding=enc)
        print(f"Cargado con encoding: {enc}")
        break
    except Exception as e:
        print(f"{enc}: {str(e)[:80]}")
        continue

print(f"\nDimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nColumnas detectadas:")
for i, c in enumerate(df.columns):
    print(f"   [{i}] {repr(c)}")

Cargado con encoding: utf-8-sig

Dimensiones: 34,656 filas × 15 columnas

Columnas detectadas:
   [0] '_id'
   [1] 'Anio'
   [2] 'ID region'
   [3] 'Region'
   [4] 'Provincia'
   [5] 'Comuna'
   [6] 'Numero explotacion'
   [7] 'Numero bloque'
   [8] 'Tipo productor'
   [9] 'Especie'
   [10] 'Variedad'
   [11] 'Anio plantacion'
   [12] 'Metodo de riego'
   [13] 'Numero de arboles'
   [14] 'Superficie (ha)'


In [9]:
# ============================================================
# 3. FILTRAR SOLO PROVINCIA COLCHAGUA
# ============================================================
df['Provincia_norm'] = df['Provincia'].astype(str).str.strip().str.upper()

print("Provincias únicas en el archivo:")
print(df['Provincia_norm'].value_counts())

df_colchagua = df[df['Provincia_norm'] == 'COLCHAGUA'].copy()
df_colchagua = df_colchagua.drop(columns=['Provincia_norm'])

print(f"\nFilas de Colchagua: {len(df_colchagua):,}")
print(f"   Comunas: {sorted(df_colchagua['Comuna'].unique())}")

Provincias únicas en el archivo:
Provincia_norm
CACHAPOAL        22375
COLCHAGUA        11269
CARDENAL CARO     1012
Name: count, dtype: int64

Filas de Colchagua: 11,269
   Comunas: ['Chimbarongo', 'Chépica', 'Lolol', 'Nancagua', 'Palmilla', 'Peralillo', 'Placilla', 'Pumanque', 'San Fernando', 'Santa Cruz']


In [10]:
# ============================================================
# 4. ELIMINAR COLUMNAS QUE NO SIRVEN
# ============================================================
columnas_a_eliminar = ['_id', 'ID region', 'Provincia', 'Region']

cols_existentes = [c for c in columnas_a_eliminar if c in df_colchagua.columns]
print(f"Columnas a eliminar: {cols_existentes}")

df_colchagua = df_colchagua.drop(columns=cols_existentes)

print(f"\nColumnas restantes ({len(df_colchagua.columns)}):")
for c in df_colchagua.columns:
    print(f"   - {c}")

Columnas a eliminar: ['_id', 'ID region', 'Provincia', 'Region']

Columnas restantes (11):
   - Anio
   - Comuna
   - Numero explotacion
   - Numero bloque
   - Tipo productor
   - Especie
   - Variedad
   - Anio plantacion
   - Metodo de riego
   - Numero de arboles
   - Superficie (ha)


In [11]:
# ============================================================
# 5. LIMPIAR Superficie (ha): coma decimal → punto
# ============================================================
if 'Superficie (ha)' in df_colchagua.columns:
    df_colchagua['Superficie (ha)'] = (
        df_colchagua['Superficie (ha)']
        .astype(str)
        .str.strip()
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )
    df_colchagua['Superficie (ha)'] = pd.to_numeric(df_colchagua['Superficie (ha)'], errors='coerce')
    print(f"Superficie convertida a float")
    print(f"   Total hectáreas Colchagua: {df_colchagua['Superficie (ha)'].sum():,.2f} ha")
    print(f"   Valores nulos: {df_colchagua['Superficie (ha)'].isna().sum()}")

Superficie convertida a float
   Total hectáreas Colchagua: 30,820.44 ha
   Valores nulos: 0


In [12]:
# ============================================================
# 6. NORMALIZAR COLUMNAS NUMÉRICAS
# ============================================================
for col in ['Anio', 'Numero explotacion', 'Numero bloque',
            'Anio plantacion', 'Numero de arboles']:
    if col in df_colchagua.columns:
        df_colchagua[col] = pd.to_numeric(df_colchagua[col], errors='coerce')

# Eliminar columnas residuales "Unnamed"
df_colchagua = df_colchagua.loc[:, ~df_colchagua.columns.str.contains('^Unnamed', na=False)]

print(f"\nTipos de datos finales:")
print(df_colchagua.dtypes)


Tipos de datos finales:
Anio                    int64
Comuna                 object
Numero explotacion      int64
Numero bloque           int64
Tipo productor         object
Especie                object
Variedad               object
Anio plantacion         int64
Metodo de riego        object
Numero de arboles       int64
Superficie (ha)       float64
dtype: object


In [13]:
# ============================================================
# 7. VALIDACIÓN
# ============================================================
print("\nValidación final:")
print(f"   Filas: {len(df_colchagua):,}")
print(f"   Comunas únicas: {df_colchagua['Comuna'].nunique()}")
print(f"   Especies únicas: {df_colchagua['Especie'].nunique()}")
print(f"   Años únicos: {sorted(df_colchagua['Anio'].dropna().unique().astype(int))}")

print(f"\nTop 10 especies por superficie:")
top_especies = (df_colchagua.groupby('Especie')['Superficie (ha)']
                .sum().sort_values(ascending=False).head(10))
print(top_especies)

print(f"\nSuperficie total por comuna:")
por_comuna = (df_colchagua.groupby('Comuna')['Superficie (ha)']
              .sum().sort_values(ascending=False))
print(por_comuna)

print(f"\nPrimeras filas finales:")
print(df_colchagua.head(10).to_string())


Validación final:
   Filas: 11,269
   Comunas únicas: 10
   Especies únicas: 31
   Años únicos: [np.int64(2021)]

Top 10 especies por superficie:
Especie
Cerezo                       9729.22
Ciruelo Europeo              5505.65
Vid de Mesa                  4373.97
Manzano Rojo                 2376.25
Nogal                        1385.48
Peral                        1219.37
Olivo                        1133.01
Kiwi                         1081.08
Manzano Verde                 621.01
Duraznero Tipo Conservero     527.81
Name: Superficie (ha), dtype: float64

Superficie total por comuna:
Comuna
Chimbarongo     7874.06
San Fernando    5414.21
Lolol           2873.20
Santa Cruz      2651.79
Palmilla        2643.79
Placilla        2586.14
Chépica         2299.56
Nancagua        2097.74
Peralillo       1786.93
Pumanque         593.02
Name: Superficie (ha), dtype: float64

Primeras filas finales:
     Anio        Comuna  Numero explotacion  Numero bloque Tipo productor       Especie          

In [14]:
# ============================================================
# 8. GUARDAR Y DESCARGAR
# ============================================================
output_name = 'odepa_colchagua_fruticola.csv'
df_colchagua.to_csv(output_name, index=False, encoding='utf-8-sig')

print(f"\nGuardado: {output_name}")
files.download(output_name)


Guardado: odepa_colchagua_fruticola.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ETL INCENDIOS

In [22]:
from google.colab import files
uploaded = files.upload()  # Selecciona tu archivo .xlsx

Saving 10.- Resumen de Ocurrencia y Daño por Comuna, 1985 - 2024_octubre.xlsx to 10.- Resumen de Ocurrencia y Daño por Comuna, 1985 - 2024_octubre.xlsx


In [23]:
import pandas as pd
from google.colab import files

file_name = '10.- Resumen de Ocurrencia y Daño por Comuna, 1985 - 2024_octubre.xlsx'

xls = pd.ExcelFile(file_name)
temporadas = ['2018-2019', '2019-2020', '2020-2021', '2021-2022']

# Mapeo de columnas reales según CONAF
columnas_reales = [
    'REGION', 'PROVINCIA', 'COMUNA', 'NUMERO_INCENDIOS',
    'PINO_INSIGNE_0_10', 'PINO_INSIGNE_11_17', 'PINO_INSIGNE_18_MAS',
    'TOTAL_PINO_INSIGNE', 'EUCALIPTO', 'OTRAS_ESPECIES',
    'TOTAL_PLANTACIONES', 'ARBOLADO', 'MATORRAL', 'PASTIZAL',
    'TOTAL_VEGETACION_NATURAL', 'TOTAL_FORESTAL',
    'AGRICOLA', 'DESECHOS', 'TOTAL_OTRAS_SUPERFICIES',
    'TOTAL_SUPERFICIE_AFECTADA'
]

hojas_seleccionadas = []
for sheet in xls.sheet_names:
    sheet_norm = sheet.replace(' ', '').replace('_', '').strip()
    for temp in temporadas:
        if temp.replace('-', '') in sheet_norm.replace('-', ''):
            hojas_seleccionadas.append((sheet, temp))
            break

dfs = []

for hoja, periodo in hojas_seleccionadas:
    raw = pd.read_excel(file_name, sheet_name=hoja, header=None)

    header_row = None
    for i in range(15):
        row_str = raw.iloc[i].astype(str).str.upper()
        if row_str.str.contains('REGION').any() and row_str.str.contains('COMUNA').any():
            header_row = i
            break
    if header_row is None:
        continue

    df = pd.read_excel(file_name, sheet_name=hoja, header=header_row)
    df.columns = columnas_reales[:len(df.columns)]

    df = df[df['COMUNA'].notna()]
    df = df[~df['COMUNA'].astype(str).str.upper().str.contains('TOTAL|NOTA|CORPORACION|GERENCIA', na=False)]
    df['REGION'] = df['REGION'].ffill()
    df['PROVINCIA'] = df['PROVINCIA'].ffill()

    df_colchagua = df[df['PROVINCIA'].astype(str).str.upper().str.strip() == 'COLCHAGUA'].copy()

    # Convertir a numérico todo excepto texto
    cols_num = [c for c in df_colchagua.columns if c not in ['REGION', 'PROVINCIA', 'COMUNA']]
    for c in cols_num:
        df_colchagua[c] = pd.to_numeric(df_colchagua[c], errors='coerce')

    # Agregar columna AÑO (año de inicio de la temporada)
    df_colchagua['ANIO'] = int(periodo.split('-')[0])

    dfs.append(df_colchagua)
    print(f"{hoja} → {len(df_colchagua)} comunas")

# Consolidar todo
df_final = pd.concat(dfs, ignore_index=True)

# Eliminar columnas REGION y PROVINCIA (redundantes)
df_final = df_final.drop(columns=['REGION', 'PROVINCIA'])

# Ordenar columnas: COMUNA, ANIO primero
orden = ['COMUNA', 'ANIO'] + [c for c in df_final.columns if c not in ['COMUNA', 'ANIO']]
df_final = df_final[orden]

# Ordenar filas por comuna y año
df_final = df_final.sort_values(['COMUNA', 'ANIO']).reset_index(drop=True)

# Guardar un solo CSV
output_name = 'colchagua_incendios_2018_2021.csv'
df_final.to_csv(output_name, index=False, encoding='utf-8-sig')

print(f"\nTotal de registros: {len(df_final)}")
print(f"Comunas: {sorted(df_final['COMUNA'].unique())}")
print(f"Años: {sorted(df_final['ANIO'].unique())}")
print(f"\nColumnas finales: {list(df_final.columns)}")

files.download(output_name)

2021 - 2022 → 10 comunas
2020 - 2021 → 10 comunas
2019-2020 → 10 comunas
2018-2019 → 10 comunas

Total de registros: 40
Comunas: ['CHEPICA', 'CHIMBARONGO', 'LOLOL', 'NANCAGUA', 'PALMILLA', 'PERALILLO', 'PLACILLA', 'PUMANQUE', 'SAN FERNANDO', 'SANTA CRUZ']
Años: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]

Columnas finales: ['COMUNA', 'ANIO', 'NUMERO_INCENDIOS', 'PINO_INSIGNE_0_10', 'PINO_INSIGNE_11_17', 'PINO_INSIGNE_18_MAS', 'TOTAL_PINO_INSIGNE', 'EUCALIPTO', 'OTRAS_ESPECIES', 'TOTAL_PLANTACIONES', 'ARBOLADO', 'MATORRAL', 'PASTIZAL', 'TOTAL_VEGETACION_NATURAL', 'TOTAL_FORESTAL', 'AGRICOLA', 'DESECHOS', 'TOTAL_OTRAS_SUPERFICIES', 'TOTAL_SUPERFICIE_AFECTADA']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
df_final.to_csv('colchagua_incendios.csv', index=False, encoding='utf-8-sig')
from google.colab import files
files.download('colchagua_incendios.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
import pandas as pd
import numpy as np
from google.colab import files

# ============================================================
# 1. CARGA DEL CSV CONSOLIDADO
# ============================================================
df = pd.read_csv('colchagua_incendios_2018_2021.csv', encoding='utf-8-sig')

print("Datos originales:")
print(f"   Filas: {len(df)} | Columnas: {len(df.columns)}")
print(f"   Comunas: {df['COMUNA'].nunique()} | Años: {df['ANIO'].nunique()}")

# ============================================================
# 2. NORMALIZACIÓN DE NOMBRES DE COLUMNAS
# ============================================================
# Renombrar a nombres más descriptivos y legibles
rename_map = {
    'COMUNA': 'comuna',
    'ANIO': 'anio',
    'NUMERO_INCENDIOS': 'n_incendios',
    'PINO_INSIGNE_0_10': 'ha_pino_0_10',
    'PINO_INSIGNE_11_17': 'ha_pino_11_17',
    'PINO_INSIGNE_18_MAS': 'ha_pino_18_mas',
    'TOTAL_PINO_INSIGNE': 'ha_pino_total',
    'EUCALIPTO': 'ha_eucalipto',
    'OTRAS_ESPECIES': 'ha_otras_especies',
    'TOTAL_PLANTACIONES': 'ha_plantaciones_total',
    'ARBOLADO': 'ha_arbolado',
    'MATORRAL': 'ha_matorral',
    'PASTIZAL': 'ha_pastizal',
    'TOTAL_VEGETACION_NATURAL': 'ha_vegetacion_natural_total',
    'TOTAL_FORESTAL': 'ha_forestal_total',
    'AGRICOLA': 'ha_agricola',
    'DESECHOS': 'ha_desechos',
    'TOTAL_OTRAS_SUPERFICIES': 'ha_otras_superficies_total',
    'TOTAL_SUPERFICIE_AFECTADA': 'ha_total_afectada'
}
df = df.rename(columns=rename_map)

# ============================================================
# 3. LIMPIEZA DE TIPOS Y VALORES
# ============================================================
# Columnas numéricas
cols_num = [c for c in df.columns if c not in ['comuna']]

# Convertir a numérico forzando errores a NaN
for c in cols_num:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Redondear a 2 decimales (evita 84.04000000000001)
df[cols_num] = df[cols_num].round(2)

# ============================================================
# 4. MANEJO DE FILAS VACÍAS (ej. PLACILLA 2020)
# ============================================================
# Si todas las métricas son NaN, asumimos 0 incendios y 0 hectáreas
metricas = [c for c in df.columns if c not in ['comuna', 'anio']]
mask_vacio = df[metricas].isna().all(axis=1)

print(f"\nFilas completamente vacías: {mask_vacio.sum()}")
if mask_vacio.any():
    print(df.loc[mask_vacio, ['comuna', 'anio']])
    df.loc[mask_vacio, metricas] = 0

# Rellenar NaN restantes con 0 (no hubo afectación en esa categoría)
df[metricas] = df[metricas].fillna(0)

# ============================================================
# 5. NORMALIZACIÓN DE TIPOS
# ============================================================
df['anio'] = df['anio'].astype(int)
df['n_incendios'] = df['n_incendios'].astype(int)

# Comuna en formato título (Chepica → Chépica, etc.)
comunas_map = {
    'CHEPICA': 'Chépica',
    'CHIMBARONGO': 'Chimbarongo',
    'LOLOL': 'Lolol',
    'NANCAGUA': 'Nancagua',
    'PALMILLA': 'Palmilla',
    'PERALILLO': 'Peralillo',
    'PLACILLA': 'Placilla',
    'PUMANQUE': 'Pumanque',
    'SAN FERNANDO': 'San Fernando',
    'SANTA CRUZ': 'Santa Cruz'
}
df['comuna'] = df['comuna'].str.upper().map(comunas_map)

# ============================================================
# 6. VALIDACIÓN DE CONSISTENCIA
# ============================================================
print("\nValidación de totales:")

# Verificar que ha_total_afectada = ha_forestal_total + ha_otras_superficies_total
check1 = (df['ha_total_afectada'] - (df['ha_forestal_total'] + df['ha_otras_superficies_total'])).abs().max()
print(f"   Diferencia máx. (total vs suma de partes): {check1:.2f} ha")

# Verificar que ha_pino_total = suma de los 3 pino
check2 = (df['ha_pino_total'] - (df['ha_pino_0_10'] + df['ha_pino_11_17'] + df['ha_pino_18_mas'])).abs().max()
print(f"   Diferencia máx. (pino total vs partes): {check2:.2f} ha")

# ============================================================
# 7. COLUMNAS DERIVADAS (para enriquecer el análisis)
# ============================================================
df['ha_promedio_por_incendio'] = np.where(
    df['n_incendios'] > 0,
    (df['ha_total_afectada'] / df['n_incendios']).round(2),
    0
)

df['pct_agricola'] = np.where(
    df['ha_total_afectada'] > 0,
    (df['ha_agricola'] / df['ha_total_afectada'] * 100).round(2),
    0
)

df['pct_forestal'] = np.where(
    df['ha_total_afectada'] > 0,
    (df['ha_forestal_total'] / df['ha_total_afectada'] * 100).round(2),
    0
)

df['pct_vegetacion_natural'] = np.where(
    df['ha_total_afectada'] > 0,
    (df['ha_vegetacion_natural_total'] / df['ha_total_afectada'] * 100).round(2),
    0
)

# ============================================================
# 8. ORDENAMIENTO FINAL
# ============================================================
df = df.sort_values(['comuna', 'anio']).reset_index(drop=True)

# Reordenar columnas: identificadores → métricas → derivadas
cols_id = ['comuna', 'anio', 'n_incendios']
cols_metricas = [
    'ha_pino_0_10', 'ha_pino_11_17', 'ha_pino_18_mas', 'ha_pino_total',
    'ha_eucalipto', 'ha_otras_especies', 'ha_plantaciones_total',
    'ha_arbolado', 'ha_matorral', 'ha_pastizal', 'ha_vegetacion_natural_total',
    'ha_forestal_total',
    'ha_agricola', 'ha_desechos', 'ha_otras_superficies_total',
    'ha_total_afectada'
]
cols_derivadas = ['ha_promedio_por_incendio', 'pct_agricola', 'pct_forestal', 'pct_vegetacion_natural']
df = df[cols_id + cols_metricas + cols_derivadas]

# ============================================================
# 9. EXPORTAR CSV LIMPIO
# ============================================================
output_name = 'colchagua_incendios_limpio.csv'
df.to_csv(output_name, index=False, encoding='utf-8-sig')

print(f"\nArchivo generado: {output_name}")
print(f"   Filas: {len(df)} | Columnas: {len(df.columns)}")
print(f"\nPrimeras filas:")
print(df.head(8).to_string())

files.download(output_name)

Datos originales:
   Filas: 40 | Columnas: 19
   Comunas: 10 | Años: 4

Filas completamente vacías: 0

Validación de totales:
   Diferencia máx. (total vs suma de partes): 0.01 ha
   Diferencia máx. (pino total vs partes): 0.00 ha

Archivo generado: colchagua_incendios_limpio.csv
   Filas: 40 | Columnas: 23

Primeras filas:
        comuna  anio  n_incendios  ha_pino_0_10  ha_pino_11_17  ha_pino_18_mas  ha_pino_total  ha_eucalipto  ha_otras_especies  ha_plantaciones_total  ha_arbolado  ha_matorral  ha_pastizal  ha_vegetacion_natural_total  ha_forestal_total  ha_agricola  ha_desechos  ha_otras_superficies_total  ha_total_afectada  ha_promedio_por_incendio  pct_agricola  pct_forestal  pct_vegetacion_natural
0  Chimbarongo  2018            5           0.0            0.0             0.0            0.0          0.90                0.0                   0.90         0.00         1.20         9.30                        10.50              11.40         0.00          0.5                        

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##ETL ECONOMIA

In [ ]:
# ============================================================
# 1. SUBIR AMBOS ARCHIVOS
# ============================================================
from google.colab import files
print("Sube 'Datamart_Economico_Colchagua.csv' y 'PUB_COMU_ACT_V2.TXT':")
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 2. CARGAR DATAMART EXISTENTE (2018-2020)
# ============================================================
df_datamart = pd.read_csv('Datamart_Economico_Colchagua.csv')
print(f"Datamart original: {len(df_datamart)} filas")
print(f"   Años: {sorted(df_datamart['Anio'].unique())}")

In [ ]:
# ============================================================
# 3. CARGAR TXT DEL SII (latin-1, tab)
# ============================================================
df_sii = pd.read_csv(
    'PUB_COMU_ACT_V2.txt',
    sep='\t',
    encoding='latin-1',
    dtype=str,
    on_bad_lines='skip',
    engine='python'
)

print(f"Archivo SII cargado: {df_sii.shape[0]:,} filas × {df_sii.shape[1]} columnas")
print(f"\nColumnas:")
for i, c in enumerate(df_sii.columns):
    print(f"   [{i}] {c}")

In [ ]:
# ============================================================
# 4. FILTRAR POR AÑO 2021 Y REGIÓN O'HIGGINS (antes de procesar todo)
# ============================================================
df_sii['Año Comercial'] = pd.to_numeric(df_sii['Año Comercial'], errors='coerce')
df_sii = df_sii[df_sii['Año Comercial'] == 2021].copy()
print(f"Filas año 2021: {len(df_sii):,}")

# Filtrar región O'Higgins
col_region = 'Región del domicilio o casa matriz'
df_sii = df_sii[df_sii[col_region].str.contains("O'HIGGINS|OHIGGINS|O’HIGGINS", case=False, na=False)].copy()
print(f"Filas 2021 en O'Higgins: {len(df_sii):,}")

In [ ]:
# ============================================================
# 5. SELECCIONAR COLUMNAS DE INTERÉS
# ============================================================
columnas_sii = {
    'Año Comercial': 'Anio',
    'Comuna del domicilio o casa matriz': 'Comuna',
    'Actividad económica': 'Actividad_Agricola',
    'Número de empresas': 'Cantidad_Empresas',
    'Ventas anuales en UF': 'Ventas_Totales_UF',
    'Número de trabajadores dependientes informados': 'Trabajadores_Dependientes'
}

df_2021 = df_sii[list(columnas_sii.keys())].rename(columns=columnas_sii).copy()

In [ ]:
# ============================================================
# 6. FILTRAR SOLO LAS 10 COMUNAS DE COLCHAGUA
# ============================================================
comunas_colchagua = [
    'SAN FERNANDO', 'CHIMBARONGO', 'NANCAGUA', 'PLACILLA', 'SANTA CRUZ',
    'CHEPICA', 'PALMILLA', 'PERALILLO', 'PUMANQUE', 'LOLOL'
]

# Normalizar comuna (mayúsculas sin tildes)
df_2021['Comuna_norm'] = (df_2021['Comuna']
                          .astype(str).str.upper().str.strip()
                          .str.replace('Á','A').str.replace('É','E')
                          .str.replace('Í','I').str.replace('Ó','O')
                          .str.replace('Ú','U'))

df_2021 = df_2021[df_2021['Comuna_norm'].isin(comunas_colchagua)].copy()
df_2021 = df_2021.drop(columns=['Comuna_norm'])

print(f"\nFilas de Colchagua 2021: {len(df_2021):,}")
print(f"   Comunas: {sorted(df_2021['Comuna'].unique())}")

In [ ]:
# ============================================================
# 7. LIMPIAR NÚMEROS (punto = separador de miles) Y TIPOS
# ============================================================
def limpiar_numero(x):
    """Convierte '3.520.604' → 3520604. Maneja NaN y strings vacíos."""
    if pd.isna(x):
        return 0
    s = str(x).strip().replace('.', '').replace(',', '.')
    try:
        return int(float(s))
    except:
        return 0

for col in ['Cantidad_Empresas', 'Ventas_Totales_UF', 'Trabajadores_Dependientes']:
    df_2021[col] = df_2021[col].apply(limpiar_numero)

# Año como int
df_2021['Anio'] = df_2021['Anio'].astype(int)

# Comuna en formato consistente (mayúsculas sin tildes, igual que el datamart original)
df_2021['Comuna'] = (df_2021['Comuna'].astype(str).str.upper().str.strip()
                     .str.replace('Á','A').str.replace('É','E')
                     .str.replace('Í','I').str.replace('Ó','O')
                     .str.replace('Ú','U'))

# Actividad: quitar espacios extra
df_2021['Actividad_Agricola'] = df_2021['Actividad_Agricola'].astype(str).str.strip()

print(f"\nEstructura de df_2021:")
print(df_2021.dtypes)
print(f"\nPrimeras filas:")
print(df_2021.head(10).to_string())

In [ ]:
# ============================================================
# 8. CONCATENAR CON EL DATAMART ORIGINAL
# ============================================================
# Asegurar que Anio sea int en ambos
df_datamart['Anio'] = df_datamart['Anio'].astype(int)

# Si ya existía 2021 en el datamart, eliminarlo para reemplazar
if 2021 in df_datamart['Anio'].unique():
    print("Se elimina 2021 preexistente del datamart original.")
    df_datamart = df_datamart[df_datamart['Anio'] != 2021]

df_final = pd.concat([df_datamart, df_2021], ignore_index=True)
df_final = df_final.sort_values(['Comuna', 'Anio', 'Actividad_Agricola']).reset_index(drop=True)

print(f"\nDatamart final: {len(df_final):,} filas")
print(f"   Años: {sorted(df_final['Anio'].unique())}")
print(f"   Comunas: {sorted(df_final['Comuna'].unique())}")
print(f"\nFilas por año:")
print(df_final.groupby('Anio').size())

In [ ]:
# ============================================================
# 9. VALIDACIÓN
# ============================================================
print("\nValidación:")
print(f"   Comunas únicas: {df_final['Comuna'].nunique()} (esperado: 10)")
print(f"   Años únicos: {df_final['Anio'].nunique()} (esperado: 4)")
print(f"\nTotales por año:")
resumen = df_final.groupby('Anio').agg(
    empresas=('Cantidad_Empresas', 'sum'),
    ventas_UF=('Ventas_Totales_UF', 'sum'),
    trabajadores=('Trabajadores_Dependientes', 'sum')
)
print(resumen)

print(f"\n   Nulos por columna:")
print(df_final[['Anio','Comuna','Actividad_Agricola','Cantidad_Empresas',
                'Ventas_Totales_UF','Trabajadores_Dependientes']].isna().sum())

In [ ]:
# ============================================================
# 10. GUARDAR Y DESCARGAR
# ============================================================
output_name = 'Datamart_Economico_Colchagua_2018_2021.csv'
df_final.to_csv(output_name, index=False, encoding='utf-8-sig')

print(f"\nArchivo guardado: {output_name}")
files.download(output_name)

In [ ]:
import pandas as pd

# Cargar solo unas filas para ver qué valores tiene la columna Región
df_test = pd.read_csv('PUB_COMU_ACT_V2.txt', sep='\t', encoding='latin-1',
                     dtype=str, nrows=5000, engine='python')

print("Valores únicos de la columna Región:")
print(df_test['Región del domicilio o casa matriz'].value_counts().head(20))

print("\nValores únicos de la columna Provincia:")
print(df_test['Provincia del domicilio o casa matriz'].value_counts().head(20))

In [ ]:
# ============================================================
# 3. CARGAR TXT DEL SII (latin-1, tab)
# ============================================================
df_sii = pd.read_csv(
    'PUB_COMU_ACT_V2.txt',
    sep='\t',
    encoding='latin-1',
    dtype=str,
    on_bad_lines='skip',
    engine='python'
)

print(f"Archivo SII: {df_sii.shape[0]:,} filas")

# ============================================================
# 4. FILTRAR AÑO 2021 PRIMERO (reduce mucho el dataset)
# ============================================================
df_sii['Año Comercial'] = pd.to_numeric(df_sii['Año Comercial'], errors='coerce')
df_sii = df_sii[df_sii['Año Comercial'] == 2021].copy()
print(f"Filas 2021: {len(df_sii):,}")

# ============================================================
# 5. FILTRAR POR REGIÓN O'HIGGINS (robusto)
# ============================================================
col_region = 'Región del domicilio o casa matriz'

# Normalizar: mayúsculas + quitar tildes y apóstrofes
def normalizar(s):
    return (s.astype(str).str.upper()
            .str.replace('Á','A').str.replace('É','E').str.replace('Í','I')
            .str.replace('Ó','O').str.replace('Ú','U')
            .str.replace("'", '').str.replace("’", '').str.replace('`', '')
            .str.strip())

df_sii['region_norm'] = normalizar(df_sii[col_region])

# Ver qué valores únicos hay (útil para depurar)
print(f"\nRegiones únicas en 2021 (primeras 20):")
print(df_sii['region_norm'].value_counts().head(20))

# Filtrar O'Higgins de forma flexible
mask_ohiggins = df_sii['region_norm'].str.contains('HIGGINS', na=False)
df_sii = df_sii[mask_ohiggins].copy()
print(f"\nFilas 2021 en O'Higgins: {len(df_sii):,}")

# ============================================================
# 6. FILTRAR SOLO LAS 10 COMUNAS DE COLCHAGUA
# ============================================================
comunas_colchagua = [
    'SAN FERNANDO', 'CHIMBARONGO', 'NANCAGUA', 'PLACILLA', 'SANTA CRUZ',
    'CHEPICA', 'PALMILLA', 'PERALILLO', 'PUMANQUE', 'LOLOL'
]

col_comuna = 'Comuna del domicilio o casa matriz'
df_sii['comuna_norm'] = normalizar(df_sii[col_comuna])

# Ver qué comunas hay (para depurar)
print(f"\nComunas únicas en O'Higgins (primeras 30):")
print(sorted(df_sii['comuna_norm'].unique())[:30])

df_2021 = df_sii[df_sii['comuna_norm'].isin(comunas_colchagua)].copy()
print(f"\nFilas Colchagua 2021: {len(df_2021):,}")

if len(df_2021) == 0:
    raise ValueError("No se encontraron filas. Revisa las comunas únicas arriba y ajusta la lista.")

# ============================================================
# 7. SELECCIONAR Y RENOMBRAR COLUMNAS
# ============================================================
columnas_sii = {
    'Año Comercial': 'Anio',
    'Comuna del domicilio o casa matriz': 'Comuna',
    'Actividad económica': 'Actividad_Agricola',
    'Número de empresas': 'Cantidad_Empresas',
    'Ventas anuales en UF': 'Ventas_Totales_UF',
    'Número de trabajadores dependientes informados': 'Trabajadores_Dependientes'
}

df_2021 = df_2021[list(columnas_sii.keys())].rename(columns=columnas_sii).copy()

# ============================================================
# 8. LIMPIAR NÚMEROS (punto = separador de miles)
# ============================================================
def limpiar_numero(x):
    if pd.isna(x):
        return 0
    s = str(x).strip().replace('.', '').replace(',', '.')
    try:
        return int(float(s))
    except:
        return 0

for col in ['Cantidad_Empresas', 'Ventas_Totales_UF', 'Trabajadores_Dependientes']:
    df_2021[col] = df_2021[col].apply(limpiar_numero)

df_2021['Anio'] = df_2021['Anio'].astype(int)
df_2021['Comuna'] = normalizar(df_2021['Comuna'])
df_2021['Actividad_Agricola'] = df_2021['Actividad_Agricola'].astype(str).str.strip()

print(f"\nEjemplo de df_2021:")
print(df_2021.head(10).to_string())

In [ ]:
# ============================================================
# 9. CONCATENAR CON EL DATAMART ORIGINAL
# ============================================================
df_datamart = pd.read_csv('Datamart_Economico_Colchagua.csv')
df_datamart['Anio'] = df_datamart['Anio'].astype(int)

# Eliminar 2021 si ya existía (para reemplazar)
if 2021 in df_datamart['Anio'].unique():
    df_datamart = df_datamart[df_datamart['Anio'] != 2021]

df_final = pd.concat([df_datamart, df_2021], ignore_index=True)
df_final = df_final.sort_values(['Comuna', 'Anio', 'Actividad_Agricola']).reset_index(drop=True)

print(f"\nDatamart final: {len(df_final):,} filas")
print(f"   Años: {sorted(df_final['Anio'].unique())}")
print(f"\nFilas por año:")
print(df_final.groupby('Anio').size())

In [ ]:
# ============================================================
# 10. GUARDAR
# ============================================================
output_name = 'Datamart_Economico_Colchagua_2018_2021.csv'
df_final.to_csv(output_name, index=False, encoding='utf-8-sig')
files.download(output_name)
print(f"\nGuardado: {output_name}")

## ETL CLIMA

In [ ]:
import pandas as pd
import io
from google.colab import files

In [ ]:
import re # Librería para limpieza de texto avanzada

print("Por favor, selecciona tus archivos CSV de clima:")
archivos_subidos = files.upload()

lista_dataframes = []

for nombre_archivo in archivos_subidos.keys():
    # 1. Leer el CSV saltando las líneas de metadatos
    df = pd.read_csv(io.BytesIO(archivos_subidos[nombre_archivo]), skiprows=3)

    # 2. Convertir 'time' a fecha y filtrar
    df['time'] = pd.to_datetime(df['time'])
    mascara_fechas = (df['time'] >= '2018-01-01') & (df['time'] <= '2021-12-31')
    df_filtrado = df.loc[mascara_fechas].copy()

    # 3. Limpieza avanzada del nombre de la comuna
    nombre_comuna = nombre_archivo.replace('.csv', '').replace('_', ' ')

    # Elimina la palabra "clima" o "Clima" y cualquier espacio extra
    nombre_comuna = re.sub(r'(?i)clima\s*', '', nombre_comuna)

    # Elimina números entre paréntesis y espacios adyacentes, ej: " (3)"
    nombre_comuna = re.sub(r'\s*\(\d+\)', '', nombre_comuna)

    # Formatea la primera letra en mayúscula (ej: "chepica" -> "Chepica")
    nombre_comuna = nombre_comuna.strip().title()

    # 4. Insertar la columna limpia
    df_filtrado.insert(0, 'comuna', nombre_comuna)

    # Añadir a la lista maestra
    lista_dataframes.append(df_filtrado)

# Unir todo en un solo DataFrame
if lista_dataframes:
    df_clima_consolidado = pd.concat(lista_dataframes, ignore_index=True)
    print("\n¡Procesamiento terminado!")
    print(f"Total de registros consolidados: {len(df_clima_consolidado)}")
else:
    print("\nNo se procesó ningún archivo.")

In [ ]:
# Ver una muestra de los datos limpios para confirmar el orden de las columnas
display(df_clima_consolidado.head())

# Guardar el resultado
nombre_salida = 'clima_colchagua_2018_2021_limpios.csv'
df_clima_consolidado.to_csv(nombre_salida, index=False)

print(f"Descargando el archivo consolidado: {nombre_salida}...")

# Activar la descarga automática
files.download(nombre_salida)

## COLCHAGUA ENTREGA

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 5)

print("Setup OK")

In [ ]:
RUTA = "."

clima = pd.read_csv(f"{RUTA}/clima_colchagua_2018_2021_limpios.csv")
clima.columns = ["comuna", "time", "tmin", "tmax", "precip"]
clima["time"] = pd.to_datetime(clima["time"])
for c in ["tmin", "tmax", "precip"]:
    clima[c] = pd.to_numeric(clima[c], errors="coerce")

incendios = pd.read_csv(f"{RUTA}/colchagua_incendios_limpio.csv")

economia = pd.read_csv(f"{RUTA}/Datamart_Economico_Colchagua_2018_2021.csv")
economia.columns = ["anio", "comuna", "actividad", "n_empresas",
                    "ventas_uf", "trabajadores"]

fruticola = pd.read_csv(f"{RUTA}/odepa_colchagua_fruticola.csv")
fruticola.columns = ["anio", "comuna", "n_explotacion", "n_bloque",
                     "tipo_productor", "especie", "variedad",
                     "anio_plantacion", "metodo_riego",
                     "n_arboles", "superficie_ha"]

print("Clima     :", clima.shape)
print("Incendios :", incendios.shape)
print("Economía  :", economia.shape)
print("Frutícola :", fruticola.shape)

In [ ]:
import unicodedata

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", str(s))
                   if not unicodedata.combining(c))

def norm_comuna(series):
    s = series.astype(str).str.strip()
    s = s.apply(strip_accents)
    s = s.str.title()
    for art in [" De ", " Del ", " La ", " Las ", " Los "]:
        s = s.str.replace(art, art.lower(), regex=False)
    return s

clima["comuna_norm"]     = norm_comuna(clima["comuna"])
incendios["comuna_norm"] = norm_comuna(incendios["comuna"])
economia["comuna_norm"]  = norm_comuna(economia["comuna"])
fruticola["comuna_norm"] = norm_comuna(fruticola["comuna"])

comunas_clima = set(clima["comuna_norm"].unique())
comunas_inc   = set(incendios["comuna_norm"].unique())
comunas_eco   = set(economia["comuna_norm"].unique())
comunas_frut  = set(fruticola["comuna_norm"].unique())

print(f"Comunas clima     : {len(comunas_clima)}")
print(f"Comunas incendios : {len(comunas_inc)}")
print(f"Comunas economía  : {len(comunas_eco)}")
print(f"Comunas frutícola : {len(comunas_frut)}")
print(f"¿Todas alineadas? {comunas_clima == comunas_inc == comunas_eco == comunas_frut}")

In [ ]:
OUT = f"{RUTA}/datos_procesados"
os.makedirs(OUT, exist_ok=True)

# --- dim_comuna ---
dim_comuna = pd.DataFrame({"comuna": sorted(comunas_clima)}).reset_index(drop=True)
dim_comuna["id_comuna"] = range(1, len(dim_comuna) + 1)
dim_comuna["provincia"] = "Colchagua"
dim_comuna["region"]    = "O'Higgins"
dim_comuna["pais"]      = "Chile"
dim_comuna["comuna_norm"] = dim_comuna["comuna"]
dim_comuna = dim_comuna[["id_comuna","comuna","comuna_norm",
                          "provincia","region","pais"]]

# --- dim_tiempo ---
fechas = pd.date_range(clima["time"].min(), clima["time"].max(), freq="D")
dim_tiempo = pd.DataFrame({"fecha": fechas})
dim_tiempo["anio"]      = dim_tiempo["fecha"].dt.year
dim_tiempo["mes"]       = dim_tiempo["fecha"].dt.month
dim_tiempo["dia"]       = dim_tiempo["fecha"].dt.day
dim_tiempo["trimestre"] = dim_tiempo["fecha"].dt.quarter
dim_tiempo["id_tiempo"] = dim_tiempo["fecha"].dt.strftime("%Y%m%d").astype(int)

dim_tiempo_mes = (dim_tiempo.drop_duplicates(subset=["anio","mes"])
                  [["anio","mes","trimestre"]]
                  .sort_values(["anio","mes"]).reset_index(drop=True))
dim_tiempo_mes.insert(0, "id_tiempo_mes", range(1, len(dim_tiempo_mes)+1))

# --- dim_actividad ---
actividades = economia[["actividad"]].drop_duplicates().reset_index(drop=True)
actividades["codigo_ciiu"] = actividades["actividad"].str.extract(r"^(\d+)")
actividades["descripcion"] = actividades["actividad"].str.replace(r"^\d+\s*-\s*", "", regex=True)
actividades["id_actividad"] = range(1, len(actividades)+1)
dim_actividad = actividades[["id_actividad","codigo_ciiu","descripcion","actividad"]]

# --- dim_especie ---
especies = fruticola[["especie"]].drop_duplicates().reset_index(drop=True)
especies["id_especie"] = range(1, len(especies)+1)
dim_especie = especies[["id_especie","especie"]]

# --- dim_riego ---
riegos = fruticola[["metodo_riego"]].drop_duplicates().reset_index(drop=True)
riegos["id_riego"] = range(1, len(riegos)+1)
dim_riego = riegos[["id_riego","metodo_riego"]]

print("dim_comuna    :", dim_comuna.shape)
print("dim_tiempo    :", dim_tiempo.shape)
print("dim_tiempo_mes:", dim_tiempo_mes.shape)
print("dim_actividad :", dim_actividad.shape)
print("dim_especie   :", dim_especie.shape)
print("dim_riego     :", dim_riego.shape)

In [ ]:
# --- fact_clima_diario ---
fact_clima_diario = clima.merge(
    dim_comuna[["id_comuna","comuna_norm"]], on="comuna_norm", how="left"
)[["id_comuna","time","tmin","tmax","precip"]].rename(columns={"time":"fecha"})

fact_clima_diario["helada"]    = (fact_clima_diario["tmin"] < 0).astype(int)
fact_clima_diario["frio"]      = (fact_clima_diario["tmin"] < 7).astype(int)
fact_clima_diario["calor"]     = (fact_clima_diario["tmax"] > 30).astype(int)
fact_clima_diario["ola_calor"] = (fact_clima_diario["tmax"] > 35).astype(int)

# --- fact_clima_anual ---
fact_clima_anual = (clima.assign(anio=clima["time"].dt.year, mes=clima["time"].dt.month)
    .merge(dim_comuna[["id_comuna","comuna_norm"]], on="comuna_norm", how="left")
    .groupby(["id_comuna","anio"])
    .agg(
        tmin_prom      = ("tmin","mean"),
        tmax_prom      = ("tmax","mean"),
        tmin_min       = ("tmin","min"),
        tmax_max       = ("tmax","max"),
        precip_total   = ("precip","sum"),
        precip_max_dia = ("precip","max"),
        n_dias         = ("time","count"),
        dias_helada    = ("tmin", lambda s: (s<0).sum()),
        dias_frio      = ("tmin", lambda s: (s<7).sum()),
        dias_calor     = ("tmax", lambda s: (s>30).sum()),
        dias_ola_calor = ("tmax", lambda s: (s>35).sum()),
    ).reset_index())

# --- fact_incendios ---
fact_incendios = incendios.merge(
    dim_comuna[["id_comuna","comuna_norm"]], on="comuna_norm", how="left"
).drop(columns=["comuna","comuna_norm"])

# --- fact_economia ---
fact_economia = economia.merge(
    dim_comuna[["id_comuna","comuna_norm"]], on="comuna_norm", how="left"
).merge(
    dim_actividad[["id_actividad","actividad"]], on="actividad", how="left"
).drop(columns=["comuna","comuna_norm","actividad"])

# --- fact_fruticola ---
fact_fruticola = fruticola.merge(
    dim_comuna[["id_comuna","comuna_norm"]], on="comuna_norm", how="left"
).merge(
    dim_especie, on="especie", how="left"
).merge(
    dim_riego, on="metodo_riego", how="left"
).drop(columns=["comuna","comuna_norm","especie","metodo_riego"])

print("fact_clima_diario :", fact_clima_diario.shape)
print("fact_clima_anual  :", fact_clima_anual.shape)
print("fact_incendios    :", fact_incendios.shape)
print("fact_economia     :", fact_economia.shape)
print("fact_fruticola    :", fact_fruticola.shape)

for nombre, df in [("clima_diario", fact_clima_diario), ("incendios", fact_incendios),
                    ("economia", fact_economia), ("fruticola", fact_fruticola)]:
    print(f"  {nombre}: id_comuna NaN = {df['id_comuna'].isna().sum()}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

econ_agg = fact_economia.groupby(["id_comuna","anio"])["ventas_uf"].sum().reset_index()
econ_agg = econ_agg.merge(dim_comuna[["id_comuna","comuna"]], on="id_comuna")
pivot_ventas = econ_agg.pivot(index="comuna", columns="anio", values="ventas_uf")
pivot_ventas.plot(kind="bar", ax=axes[0,0])
axes[0,0].set_title("Ventas totales por comuna y año (UF)")
axes[0,0].tick_params(axis='x', rotation=45)

inc_anual = fact_incendios.groupby("anio")[["n_incendios","ha_total_afectada"]].sum()
inc_anual.plot(ax=axes[0,1], marker="o", secondary_y="ha_total_afectada")
axes[0,1].set_title("Incendios: cantidad vs hectáreas afectadas")

clima_agg = fact_clima_anual.groupby(["id_comuna","anio"])["dias_helada"].sum().reset_index()
clima_agg = clima_agg.merge(dim_comuna[["id_comuna","comuna"]], on="id_comuna")
pivot_hel = clima_agg.pivot(index="comuna", columns="anio", values="dias_helada")
sns.heatmap(pivot_hel, annot=True, fmt=".0f", cmap="coolwarm", ax=axes[0,2])
axes[0,2].set_title("Días de helada por comuna y año")

top_esp = (fact_fruticola.merge(dim_especie, on="id_especie")
           .groupby("especie")["superficie_ha"].sum()
           .sort_values(ascending=True).tail(10))
top_esp.plot(kind="barh", ax=axes[1,0], color="#31a354")
axes[1,0].set_title("Top 10 especies por superficie (ha)")

riego_dist = (fact_fruticola.merge(dim_riego, on="id_riego")
              .groupby("metodo_riego")["superficie_ha"].sum()
              .sort_values(ascending=False))
riego_dist.plot(kind="bar", ax=axes[1,1], color="#3182bd")
axes[1,1].set_title("Superficie por método de riego")
axes[1,1].tick_params(axis='x', rotation=45)

trab_agg = fact_economia.groupby("anio")[["n_empresas","trabajadores"]].sum()
trab_agg.plot(ax=axes[1,2], marker="o")
axes[1,2].set_title("Empresas y trabajadores totales por año")

plt.tight_layout(); plt.show()

In [ ]:
# 1) Superficie frutícola total por comuna
frut = fact_fruticola.groupby("id_comuna")["superficie_ha"].sum().rename("frut_total")

# 2) HHI de concentración de especies
sup_esp = fact_fruticola.groupby(["id_comuna","id_especie"])["superficie_ha"].sum().reset_index()
sup_esp["cuota"] = sup_esp.groupby("id_comuna")["superficie_ha"].transform(lambda s: s/s.sum())
hhi = sup_esp.groupby("id_comuna")["cuota"].apply(lambda s: (s**2).sum()).rename("hhi_monocultivo")

# 3) Incendios acumulados
inc = fact_incendios.groupby("id_comuna").agg(
    n_incendios=("n_incendios","sum"),
    ha_total_afectada=("ha_total_afectada","sum")
)

# 4) Clima anual promedio
cli = fact_clima_anual.groupby("id_comuna").agg(
    dias_helada=("dias_helada","mean"),
    precip_total=("precip_total","mean")
)

# 5) Delta ventas 2018→2021
v18 = fact_economia[fact_economia["anio"]==2018].groupby("id_comuna")["ventas_uf"].sum()
v21 = fact_economia[fact_economia["anio"]==2021].groupby("id_comuna")["ventas_uf"].sum()
delta = ((v21-v18)/v18*100).rename("delta_ventas_pct")

# Consolidar
feats = pd.concat([frut, hhi, inc, cli, delta], axis=1).reset_index()
feats = feats.merge(dim_comuna[["id_comuna","comuna"]], on="id_comuna")
print(feats.shape)
display(feats)

In [ ]:
feat_cols = ["frut_total","hhi_monocultivo","n_incendios",
             "ha_total_afectada","dias_helada","precip_total","delta_ventas_pct"]

Xs = StandardScaler().fit_transform(feats[feat_cols])

sil = {}
inercia = {}
for k in range(2, 6):
    km_k = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xs)
    sil[k] = silhouette_score(Xs, km_k.labels_)
    inercia[k] = km_k.inertia_

print("k | Inercia | Silhouette")
for k in sorted(sil):
    print(f"{k} | {inercia[k]:.2f}   | {sil[k]:.3f}")

# Guardar métricas
metricas_clustering = pd.DataFrame({
    "k": list(sil.keys()),
    "inercia": list(inercia.values()),
    "silhouette": list(sil.values())
})
metricas_clustering.to_csv(f"{OUT}/metricas_clustering.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Serie mensual de precipitación promedio provincial
serie = (fact_clima_diario.assign(
            anio=fact_clima_diario["fecha"].dt.year,
            mes=fact_clima_diario["fecha"].dt.month)
         .groupby(["anio","mes"])["precip"].mean())
serie.index = pd.date_range("2018-01-01", periods=len(serie), freq="MS")

train, test = serie[:-8], serie[-8:]   # test = últimos 8 meses

# Baseline: seasonal naive (mismo mes del año anterior)
pred_naive = serie.shift(12).loc[test.index]
rmse_naive = float(np.sqrt(((test - pred_naive)**2).mean()))
print(f"RMSE naive: {rmse_naive:.2f} mm")
print(f"Serie: {len(serie)} meses | train: {len(train)} | test: {len(test)}")

In [ ]:
sar = SARIMAX(train, order=(1,0,0), seasonal_order=(1,0,0,12),
              enforce_stationarity=False).fit(disp=False)
pred_sar = sar.forecast(len(test))

rmse_sar = float(np.sqrt(((test - pred_sar)**2).mean()))
mae_sar  = float(np.abs(test - pred_sar).mean())
smape_sar = float((2 * np.abs(test - pred_sar) /
                   (np.abs(test)+np.abs(pred_sar)).replace(0,np.nan)).mean()*100)

print(f"=== SARIMA(1,0,0)(1,0,0,12) ===")
print(f"  RMSE   : {rmse_sar:.2f} mm")
print(f"  MAE    : {mae_sar:.2f} mm")
print(f"  sMAPE  : {smape_sar:.1f} %")
print(f"  vs Naive: {100*(1 - rmse_sar/rmse_naive):+.1f}%")

In [ ]:
hw = ExponentialSmoothing(
    train,
    trend=None,
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated"
).fit(optimized=True)

pred_hw = hw.forecast(len(test))

rmse_hw = float(np.sqrt(((test - pred_hw)**2).mean()))
mae_hw  = float(np.abs(test - pred_hw).mean())
smape_hw = float((2 * np.abs(test - pred_hw) /
                  (np.abs(test)+np.abs(pred_hw)).replace(0,np.nan)).mean()*100)

print(f"=== Holt-Winters ===")
print(f"  AIC    : {hw.aic:.2f}")
print(f"  RMSE   : {rmse_hw:.2f} mm")
print(f"  MAE    : {mae_hw:.2f} mm")
print(f"  sMAPE  : {smape_hw:.1f} %")
print(f"  vs Naive: {100*(1 - rmse_hw/rmse_naive):+.1f}%")

plt.figure(figsize=(12,5))
train.plot(label="Train", color="black")
test.plot(label="Test (real)", color="blue")
pred_hw.plot(label="Holt-Winters", color="green", linestyle="--")
plt.title(f"Holt-Winters · RMSE={rmse_hw:.2f}mm · sMAPE={smape_hw:.1f}%")
plt.legend(); plt.tight_layout(); plt.show()

# Guardar métricas de forecast
metricas_forecast = pd.DataFrame({
    "modelo": ["Seasonal Naive", "SARIMA(1,0,0)(1,0,0,12)", "Holt-Winters (aditivo)"],
    "rmse_mm": [rmse_naive, rmse_sar, rmse_hw],
    "mae_mm": [np.nan, mae_sar, mae_hw],
    "smape_pct": [np.nan, smape_sar, smape_hw],
    "mejora_vs_naive_pct": [0.0, 100*(1-rmse_sar/rmse_naive), 100*(1-rmse_hw/rmse_naive)]
})
metricas_forecast.to_csv(f"{OUT}/metricas_forecast.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Reentrenar con TODA la serie
hw_full = ExponentialSmoothing(serie, trend=None, seasonal="add",
                                seasonal_periods=12,
                                initialization_method="estimated").fit(optimized=True)
fc = hw_full.forecast(24)

# IC 95% aproximado
resid_std = (serie - hw_full.fittedvalues).std()
ic_inf = fc - 1.96 * resid_std
ic_sup = fc + 1.96 * resid_std

forecast_df = pd.DataFrame({
    "fecha": fc.index,
    "forecast": fc.values,
    "ic_inf_95": ic_inf.values,
    "ic_sup_95": ic_sup.values
})
forecast_df.to_csv(f"{OUT}/forecast_precipitacion.csv", index=False, encoding="utf-8-sig")

plt.figure(figsize=(12,5))
serie.plot(label="Histórico", color="black")
fc.plot(label="Forecast 24m", color="green")
plt.fill_between(fc.index, ic_inf, ic_sup, alpha=0.2, color="green", label="IC 95%")
plt.legend(); plt.title("Forecast precipitación provincial · 24 meses")
plt.tight_layout(); plt.show()

print(forecast_df.head())

In [ ]:
k_opt = 2
km = KMeans(n_clusters=k_opt, random_state=42, n_init=10).fit(Xs)
feats["cluster"] = km.labels_

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(Xs)
feats["pc1"], feats["pc2"] = coords[:,0], coords[:,1]

plt.figure(figsize=(9,5))
sns.scatterplot(data=feats, x="pc1", y="pc2", hue="cluster",
                palette="Set1", s=180)
for _, r in feats.iterrows():
    plt.annotate(r["comuna"], (r["pc1"], r["pc2"]), fontsize=8)
plt.title(f"Clusters de comunas de Colchagua (K-Means, k={k_opt})")
plt.tight_layout(); plt.show()

print(f"\nPerfil promedio por cluster (k={k_opt}):")
display(feats.groupby("cluster")[feat_cols].mean().round(2))

print("\nComunas por cluster:")
for c in sorted(feats["cluster"].unique()):
    print(f"  Cluster {c}: {sorted(feats.loc[feats['cluster']==c,'comuna'].tolist())}")

feats["perfil"] = feats["cluster"].map({
    0: "Comuna estándar",
    1: "Comuna crítica (alta fruticultura + incendios)"
})

# Guardar clusters
feats.to_csv(f"{OUT}/vulnerabilidad_comunal.csv", index=False, encoding="utf-8-sig")
feats[["id_comuna","comuna","cluster","perfil","pc1","pc2"]].to_csv(
    f"{OUT}/clusters_comunas.csv", index=False, encoding="utf-8-sig")

In [ ]:
print("=" * 70)
print("HALLAZGOS PRINCIPALES")
print("=" * 70)

print("\n[H1] Evolución económica 2018→2021:")
for _, r in feats[["comuna","delta_ventas_pct"]].sort_values("delta_ventas_pct", ascending=False).iterrows():
    print(f"   {r['comuna']:15s} : {r['delta_ventas_pct']:+.1f}%")

print("\n[H2] Incendios acumulados 2018-2021:")
inc_tot = fact_incendios.groupby("id_comuna")[["n_incendios","ha_total_afectada"]].sum().reset_index()
inc_tot = inc_tot.merge(dim_comuna[["id_comuna","comuna"]], on="id_comuna").sort_values("ha_total_afectada", ascending=False)
for _, r in inc_tot.iterrows():
    print(f"   {r['comuna']:15s} : {int(r['n_incendios'])} incendios, {r['ha_total_afectada']:.1f} ha")

print("\n[H3] Días helada promedio 2018-2021:")
hel = fact_clima_anual.groupby("id_comuna")["dias_helada"].mean().reset_index()
hel = hel.merge(dim_comuna[["id_comuna","comuna"]], on="id_comuna").sort_values("dias_helada", ascending=False)
for _, r in hel.iterrows():
    print(f"   {r['comuna']:15s} : {r['dias_helada']:.2f} días/año")

print("\n[H4] Concentración frutícola (HHI):")
for _, r in feats[["comuna","hhi_monocultivo","frut_total"]].sort_values("hhi_monocultivo", ascending=False).iterrows():
    print(f"   {r['comuna']:15s} : HHI={r['hhi_monocultivo']:.3f}, {r['frut_total']:.0f} ha")

print("\n[H5] Clusters K-Means:")
for c in sorted(feats["cluster"].unique()):
    print(f"   Cluster {c}: {sorted(feats.loc[feats['cluster']==c,'comuna'].tolist())}")

In [ ]:
export_map = {
    # Dimensiones
    "dim_comuna.csv":            dim_comuna,
    "dim_tiempo.csv":            dim_tiempo,
    "dim_tiempo_mes.csv":        dim_tiempo_mes,
    "dim_actividad.csv":         dim_actividad,
    "dim_especie.csv":           dim_especie,
    "dim_riego.csv":             dim_riego,
    # Hechos
    "fact_clima_diario.csv":     fact_clima_diario,
    "fact_clima_anual.csv":      fact_clima_anual,
    "fact_incendios.csv":        fact_incendios,
    "fact_economia.csv":         fact_economia,
    "fact_fruticola.csv":        fact_fruticola,
    # Analítica
    "clusters_comunas.csv":      feats[["id_comuna","comuna","cluster","perfil","pc1","pc2"]],
    "vulnerabilidad_comunal.csv": feats,
    "forecast_precipitacion.csv": forecast_df,
    "metricas_clustering.csv":   metricas_clustering,
    "metricas_forecast.csv":     metricas_forecast,
}

for fname, df in export_map.items():
    df.to_csv(f"{OUT}/{fname}", index=False, encoding="utf-8-sig")
    print(f"  ✓ {fname:32s} {df.shape[0]:>6} filas × {df.shape[1]:>2} cols")

print(f"\nTotal: {len(export_map)} archivos en {OUT}/")

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/datamart_colchagua", "zip", OUT)
files.download("/content/datamart_colchagua.zip")

print("✓ Descargando datamart_colchagua.zip")
print(f"  Contiene {len(os.listdir(OUT))} archivos:")
for f in sorted(os.listdir(OUT)):
    print(f"    · {f}")

In [ ]:
# ============================================================
# FUENTES DE DATOS · citas y justificación
# ============================================================
"""
Fuentes oficiales utilizadas (todas públicas y trazables):

1. SII — Estadísticas de empresas por comuna y actividad (CIIU 4 dígitos)
   https://www.sii.cl/estadisticas
   Rol: economía PYME (ventas, empleo, n° empresas). 2018-2021.
   Justificación: única fuente oficial con desagregación comunal por CIIU.

2. CIREN / ODEPA — Catastro Frutícola Nacional 2021
   https://www.ciren.cl
   Rol: superficie frutícola, especie, variedad, riego.
   Justificación: único levantamiento nacional con detalle por especie y riego.

3. CONAF — Estadísticas de incendios forestales 2018-2021
   https://www.conaf.cl
   Rol: hectáreas afectadas por tipo de vegetación.
   Justificación: fuente oficial con series comunales desde 2018.

4. Open-Meteo — Reanálisis ERA5 (series diarias)
   https://open-meteo.com
   Rol: T min/max, precipitación. 2018-2021.
   Justificación: ERA5 diario por coordenada, permite series por comuna.

Fuentes descartadas:
- DGA: sin granularidad comunal en el periodo requerido.
- Granizo: sin fuente oficial con cobertura sistemática.
- INE exportaciones: resolución regional, no comunal (usado como contexto macro en informe).
"""
print("Fuentes citadas OK")

In [ ]:
# ============================================================
# PLANTEAMIENTO ABIERTO · hipótesis de trabajo
# ============================================================
"""
Pregunta central:
¿Qué comunas, sectores y PYMEs de Colchagua son más vulnerables a que
un shock agrícola se propague sobre su economía, y qué amenazas
—climáticas y de mercado— resultan más determinantes según la evidencia?

Enfoque: NO se cierra el problema a una amenaza específica. Se incorporan
6 dimensiones (incendios, heladas, olas de calor, precipitación, monocultivo,
estructura económica) y el análisis de los datos determina cuáles pesan más.

Hipótesis de trabajo:
  H1. Las comunas con mayor concentración frutícola son las más vulnerables
      a shocks de monocultivo.
  H2. Las amenazas climáticas tienen impacto asimétrico entre comunas.
  H3. Existe vínculo medible entre amenazas y desempeño económico PYME.

Criterio de decisión: silhouette para k óptimo, RMSE vs baseline para forecast,
HHI como proxy objetivo de monocultivo.
"""
print("Planteamiento abierto definido")